In [1]:
import sys
sys.path.append("..") # Adds the root directory to search path for imports

from src.preprocessing.data_cleaning import clean_data

# Run data cleaning
raw_data_path = "../data/raw/UA.csv"
output_dir = "../data/processed"

recommended_dataset_path = clean_data(raw_data_path, output_dir)


Loaded raw dataset from ../data/raw/UA.csv: (1537, 40)
Dropped rows with missing values. Cleaned shape: (1463, 40)
Saved full cleaned dataset to: ../data/processed\cleaned_UA.csv
Selected recommended features. Saved to ../data/processed\reccomended_features_dataset.csv: (1463, 8)


In [2]:
from src.feature_engineering.feature_engineering import engineer_dataset_from_file

# Run feature engineering on the cleaned recommended dataset
engineered_dataset_path = "../data/processed/reccomended_features_dataset_engineered.csv"

engineer_dataset_from_file(recommended_dataset_path, engineered_dataset_path)


Feature engineering complete!
Original shape: (1463, 8) -> Engineered shape: (1463, 15)
New columns added: ['BMI_Underweight', 'BMI_Normal', 'BMI_Overweight', 'BMI_Obese', 'Postmenopausal_Risk', 'Is_Senior', 'Lifestyle_Score']
Saved engineered dataset to: ../data/processed/reccomended_features_dataset_engineered.csv


'../data/processed/reccomended_features_dataset_engineered.csv'

In [3]:
from src.preprocessing.train_test_split import split_dataset

# Split the engineered dataset
train_path, test_path = split_dataset(
    dataset_path=engineered_dataset_path,
    output_dir="../data/processed",
    target_col="Fracture",
    test_size=0.1,
    random_state=42
)


Loaded dataset from ../data/processed/reccomended_features_dataset_engineered.csv for splitting: (1463, 15)
Saved stratified train split to ../data/processed\train_reccomended_features_dataset_engineered.csv: (1316, 15)
Saved stratified test split to ../data/processed\test_reccomended_features_dataset_engineered.csv: (147, 15)


In [ ]:
import pandas as pd
from src.preprocessing.pipeline import build_preprocessing_pipeline, save_pipeline

# 1. Load the splits
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Separate features and targets
X_train = train_df.drop(columns=["Fracture"])
y_train = train_df["Fracture"]

X_test = test_df.drop(columns=["Fracture"])
y_test = test_df["Fracture"]

# 2. Build the unified preprocessing pipeline
# Note: Gender is encoded, Age/Height/Weight/BMI are capped and scaled
pipeline = build_preprocessing_pipeline(
    num_cols=["Age", "Height", "Weight", "BMI"], 
    gender_col="Gender"
)

# 3. Fit pipeline on train data and transform both train and test
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

# 4. Convert back to DataFrames with columns in correct pipeline order
# Scaled features go first, followed by all passthrough features
processed_columns = [
    "Age", "Height", "Weight", "BMI",               # Scaled columns
    "Gender", "Smoking", "Drinking",                # Original passthrough columns
    "BMI_Underweight", "BMI_Normal", 
    "BMI_Overweight", "BMI_Obese",                  # Engineered passthrough columns
    "Postmenopausal_Risk", "Is_Senior", 
    "Lifestyle_Score"
]

train_df_processed = pd.concat([
    pd.DataFrame(X_train_processed, columns=processed_columns),
    y_train.reset_index(drop=True)
], axis=1)

test_df_processed = pd.concat([
    pd.DataFrame(X_test_processed, columns=processed_columns),
    y_test.reset_index(drop=True)
], axis=1)

# 5. Save the final scaled datasets
train_df_processed.to_csv("../data/processed/train_reccomended_features_dataset_engineered_scaled.csv", index=False)
test_df_processed.to_csv("../data/processed/test_reccomended_features_dataset_engineered_scaled.csv", index=False)
print("Saved final scaled train/test datasets!")

# 6. Save the fitted pipeline model for deployment
save_pipeline(pipeline, "../outputs/models/preprocessing_pipeline.joblib")


Saved final scaled train/test datasets!
Fitted pipeline saved successfully to ../outputs/models/preprocessing_pipeline.joblib
